# Lecture 1: Introduction and Supervised Learning


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression, make_classification, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score

np.random.seed(42)
print("Libraries loaded.")


## 1. Learning Paradigms

The textbook defines three fundamental learning paradigms:

| Paradigm | Description | Example |
|---|---|---|
| **Supervised** | Labeled data → learn f(x) = y | Classification, Regression |
| **Unsupervised** | Unlabeled data → discover structure | Clustering, Generative models |
| **Reinforcement** | Agent + environment → maximize reward | Game playing, Robotics |


In [ ]:
# ----- SUPERVISED LEARNING example: Boston-like regression -----
X, y = make_regression(n_samples=200, n_features=1, noise=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(X_train, y_train, alpha=0.5, label="Training data", color='steelblue')
axes[0].scatter(X_test, y_test, alpha=0.7, label="Test data", color='orange')
x_line = np.linspace(X.min(), X.max(), 100).reshape(-1,1)
axes[0].plot(x_line, model.predict(x_line), 'r-', lw=2, label="Model prediction")
axes[0].set_title("Supervised Learning: Linear Regression")
axes[0].set_xlabel("x"); axes[0].set_ylabel("y")
axes[0].legend()

# residuals
residuals = y_test - y_pred
axes[1].hist(residuals, bins=20, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title(f"Residuals — MSE: {mean_squared_error(y_test, y_pred):.1f}")
axes[1].set_xlabel("Residual (y_test - y_pred)")

plt.tight_layout()
plt.savefig("fig_nb01_supervised.png", dpi=100, bbox_inches='tight')
plt.show()
print(f"Coefficient: {model.coef_[0]:.3f} | Intercept: {model.intercept_:.3f}")


## 2. Components of Supervised Learning (Ch. 2)

In Prince's textbook, supervised learning is defined as:

> **Goal:** Learn a function `f_θ(x)` that predicts output `y` given input `x`.

Core components:
1. **Model** `f_θ(x)` — parameterized function
2. **Loss** `L(y, f_θ(x))` — measures prediction error
3. **Optimization** — updates θ to minimize the loss


In [ ]:
# ----- Manually simulate the supervised learning loop -----
# Simple model: y = w*x + b (1D linear)

np.random.seed(0)
X_raw = np.linspace(-3, 3, 100)
y_true = 2.5 * X_raw + 1.0 + np.random.normal(0, 1, 100)

# Learn w, b via manual gradient descent
w, b = 0.0, 0.0
lr = 0.01
losses = []

for epoch in range(300):
    y_hat = w * X_raw + b
    loss = np.mean((y_hat - y_true)**2)
    losses.append(loss)
    # gradients
    dw = np.mean(2 * (y_hat - y_true) * X_raw)
    db = np.mean(2 * (y_hat - y_true))
    w -= lr * dw
    b -= lr * db

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(losses, color='crimson')
axes[0].set_title("Training Loss (MSE) — Gradient Descent")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE Loss")
axes[0].set_yscale('log')

axes[1].scatter(X_raw, y_true, alpha=0.4, label="True data", color='steelblue')
axes[1].plot(X_raw, w * X_raw + b, 'r-', lw=2, label=f"Learned: y={w:.2f}x+{b:.2f}")
axes[1].plot(X_raw, 2.5*X_raw+1.0, 'g--', lw=1.5, label="True: y=2.5x+1.0")
axes[1].set_title("Regression via Manual Gradient Descent")
axes[1].legend()

plt.tight_layout()
plt.savefig("fig_nb01_gd.png", dpi=100, bbox_inches='tight')
plt.show()
print(f"Learned: w={w:.4f}, b={b:.4f}")
print(f"True:    w=2.5000, b=1.0000")


## 3. Unsupervised Learning: Clustering Example

In unsupervised learning there are no labels; the model organizes data on its own.


In [ ]:
from sklearn.cluster import KMeans

# Generate synthetic data (3 clusters)
X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, cluster_std=1.2, random_state=42)

kmeans = KMeans(n_clusters=3, random_state=42, n_init='auto')
labels = kmeans.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], alpha=0.6, color='gray')
axes[0].set_title("Unsupervised: Raw Data (no labels)")

scatter = axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1],
                          c=labels, cmap='tab10', alpha=0.7)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
               marker='X', s=200, c='black', zorder=5, label='Centroid')
axes[1].set_title("K-Means Clustering (k=3)")
axes[1].legend()

plt.tight_layout()
plt.savefig("fig_nb01_clustering.png", dpi=100, bbox_inches='tight')
plt.show()


## 4. Reinforcement Learning: Simple Grid World

In reinforcement learning an **agent** takes actions in an **environment** to collect **rewards**.


In [ ]:
# Simple 1D reward environment simulation
# Agent moves right or left at each step, trying to reach the goal (position=9)

def simple_rl_demo(n_episodes=500, epsilon=0.3, lr=0.5, gamma=0.9):
    n_states = 10
    Q = np.zeros(n_states)  # Q-values (single action: move right or stay)
    rewards_per_ep = []
    
    for ep in range(n_episodes):
        state = 0
        total_reward = 0
        for step in range(20):
            # Epsilon-greedy policy
            if np.random.rand() < epsilon:
                action = np.random.choice([-1, 1])
            else:
                action = 1  # always move right (simple)
            
            next_state = max(0, min(n_states-1, state + action))
            reward = 10 if next_state == n_states-1 else -0.1
            
            # Update Q
            Q[state] = Q[state] + lr * (reward + gamma * Q[next_state] - Q[state])
            state = next_state
            total_reward += reward
            if next_state == n_states-1:
                break
        
        rewards_per_ep.append(total_reward)
    
    return rewards_per_ep, Q

rewards, Q_final = simple_rl_demo()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Smoothed reward
smoothed = np.convolve(rewards, np.ones(30)/30, mode='valid')
axes[0].plot(smoothed, color='darkorange')
axes[0].set_title("Total Reward per Episode (smoothed)")
axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Total Reward")

axes[1].bar(range(len(Q_final)), Q_final, color='steelblue')
axes[1].set_title("Learned Q-Values (per state)")
axes[1].set_xlabel("State"); axes[1].set_ylabel("Q(s)")

plt.tight_layout()
plt.savefig("fig_nb01_rl.png", dpi=100, bbox_inches='tight')
plt.show()


## 5. Summary

| Concept | Definition |
|---|---|
| Supervised Learning | Learning f(x)→y from labeled data |
| Unsupervised Learning | Discovering structure in unlabeled data |
| Reinforcement Learning | Reward maximization via trial and error |
| Model Parameters θ | Values updated during the learning process |
| Loss Function | Measures the discrepancy between prediction and ground truth |
| Gradient Descent | Optimization algorithm that minimizes the loss |

